# E4 v3 — stability by feature-selection family

One Kaggle session runs one feature-selection family across the listed cohorts. Set `N_SUBSAMPLES=3` only for a smoke test; the protocol result requires 30. To resume, attach the prior output ZIP as a Kaggle input and set `RESUME_E4_ARCHIVE`.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile

INPUT_V3_DATASET = Path('/kaggle/input/experiment-data-v3')
REPO_URL = 'https://github.com/datachain-uit/RNA-Seq-FS-Survey.git'
REPO_BRANCH = 'survey-v2-code'
WORK_ROOT = Path('/kaggle/working/kltn_data')
# Attach the previous Kaggle output ZIP, then set this exact path for a resume run.
RESUME_E4_ARCHIVE = None  # e.g. Path('/kaggle/input/e4-embedded-partial/e4_embedded_all_cohorts.zip')

CLASSIFICATION_GROUP = 'filter'  # filter | embedded | wrappers
COHORTS = ['tcga_luad_lusc_primary_tumor', 'gse10245', 'gse30219', 'gse37745', 'gse42127']
FRACTIONS = [0.5, 0.7, 0.9]
N_SUBSAMPLES = 30
K_GRID = [1, 2, 3, 4, 5, 10, 25, 50]
METHODS_BY_GROUP = {'filter':['f_test','mutual_info','mrmr','relieff'], 'embedded':['l1_lr','elastic_net','rf_importance','svm_rfe'], 'wrappers':['ga','bpso']}
assert CLASSIFICATION_GROUP in METHODS_BY_GROUP
METHODS = METHODS_BY_GROUP[CLASSIFICATION_GROUP]
if CLASSIFICATION_GROUP == 'wrappers':
    FRACTIONS, K_GRID = [0.7], [50]  # protocol-limited wrapper stability


In [ ]:
REPO_DIR = Path('/kaggle/working/RNA-Seq-FS-Survey')
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)], check=True)
# Do not install requirements.txt on Kaggle: the base scientific stack is already compatible.
RUNTIME_IMPORTS = ('numpy','pandas','scipy','sklearn','tables','joblib','threadpoolctl')
for package in RUNTIME_IMPORTS: print(package, __import__(package).__version__)
try:
    import skrebate
except ModuleNotFoundError:
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','skrebate==0.8.4'],check=True)
WORK_ROOT.mkdir(parents=True,exist_ok=True)
for folder in ('experiment-data-perm','experiment-data-geo-target','split_manifests'):
    source,target=INPUT_V3_DATASET/folder,WORK_ROOT/folder
    assert source.is_dir(), source
    if not target.exists(): target.symlink_to(source,target_is_directory=True)

# Restore resume checkpoint CSVs only; ignore the Git clone accidentally bundled in old E4 ZIPs.
if RESUME_E4_ARCHIVE is not None:
    RESUME_E4_ARCHIVE = Path(RESUME_E4_ARCHIVE)
    assert RESUME_E4_ARCHIVE.is_file(), RESUME_E4_ARCHIVE
    resume_root = WORK_ROOT / 'results' / 'survey_v3' / 'E4_stability' / CLASSIFICATION_GROUP
    prefixes = (f'kltn_data/results/survey_v3/E4_stability/{CLASSIFICATION_GROUP}/', f'results/survey_v3/E4_stability/{CLASSIFICATION_GROUP}/')
    copied = 0
    with zipfile.ZipFile(RESUME_E4_ARCHIVE) as archive:
        for info in archive.infolist():
            name = info.filename.replace('\\', '/')
            prefix = next((item for item in prefixes if name.startswith(item)), None)
            if prefix is None:
                continue
            relative = name[len(prefix):]
            if not relative.startswith('tcga_luad_lusc_primary_tumor/checkpoints/') or not relative.endswith('.csv'):
                continue
            destination = resume_root / relative
            if destination.exists():
                continue
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, destination.open('wb') as target:
                shutil.copyfileobj(source, target)
            copied += 1
    print(f'Restored {copied} resume checkpoint CSVs from {RESUME_E4_ARCHIVE.name}')

os.environ['KLTN_DATA_ROOT']=str(WORK_ROOT)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO_DIR,text=True).strip()


In [ ]:
os.chdir(REPO_DIR); sys.path.insert(0,str(REPO_DIR))
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
from survey_v2 import config as C
from survey_v2.data import load_dataset, sha256, variance_prefilter, stratified_subsample
from survey_v2.methods import select
from survey_v2.stability import nogueira, selections_to_matrix

def save(df,path):
    path.parent.mkdir(parents=True,exist_ok=True); tmp=path.with_suffix('.tmp'); df.to_csv(tmp,index=False); tmp.replace(path)

for COHORT in COHORTS:
    ds=load_dataset(WORK_ROOT,COHORT)
    out=WORK_ROOT/'results'/'survey_v3'/'E4_stability'/CLASSIFICATION_GROUP/COHORT
    for fraction in FRACTIONS:
        for subsample in range(N_SUBSAMPLES):
            idx=stratified_subsample(ds.y,fraction,np.random.default_rng([C.dataset_seed_id(COHORT),4,int(fraction*100),subsample]))
            pre=variance_prefilter(ds.X[idx],5000)
            for method_id,method in enumerate(METHODS):
                path=out/'checkpoints'/f'f{int(fraction*100)}_s{subsample}_{method}.csv'
                if path.is_file(): continue
                with threadpool_limits(1):
                    chosen=select(method,ds.X[idx][:,pre],ds.y[idx],K_GRID,seed=C.seed_for(C.dataset_seed_id(COHORT),4,int(fraction*100),subsample,method_id),n_jobs=1)
                rows=[]
                for k in K_GRID:
                    cols=pre[chosen.panels[k]]
                    assert len(cols)==k and len(set(cols.tolist()))==k
                    rows.extend({'cohort':COHORT,'family':CLASSIFICATION_GROUP,'fraction':fraction,'subsample':subsample,'method':method,'k':k,'rank':rank,'gene':ds.genes[col]} for rank,col in enumerate(cols,1))
                save(pd.DataFrame(rows),path)
                print(f'{COHORT} {method} f={fraction} subsample={subsample}: done',flush=True)


In [ ]:
root=WORK_ROOT/'results'/'survey_v3'/'E4_stability'/CLASSIFICATION_GROUP
for COHORT in COHORTS:
    ds=load_dataset(WORK_ROOT,COHORT); out=root/COHORT
    files=sorted((out/'checkpoints').glob('*.csv'))
    expected=len(FRACTIONS)*N_SUBSAMPLES*len(METHODS)
    assert len(files)==expected,(COHORT,len(files),expected)
    genes=pd.concat([pd.read_csv(p) for p in files],ignore_index=True)
    save(genes,out/'e4_selected_genes.csv')
    rows=[]
    for (fraction,method,k),g in genes.groupby(['fraction','method','k']):
        panels=[x.gene.tolist() for _,x in g.groupby('subsample')]
        rows.append({'cohort':COHORT,'family':CLASSIFICATION_GROUP,'fraction':fraction,'method':method,'k':k,**nogueira(selections_to_matrix(panels,ds.genes))})
    save(pd.DataFrame(rows),out/'e4_stability.csv')
    (out/'config.json').write_text(json.dumps({'experiment':'E4_stability_v3','cohort':COHORT,'family':CLASSIFICATION_GROUP,'fractions':FRACTIONS,'n_subsamples':N_SUBSAMPLES,'k_grid':K_GRID,'methods':METHODS,'commit_hash':COMMIT,'traceable':True},indent=2),encoding='utf-8')
archive=Path('/kaggle/working')/f'e4_{CLASSIFICATION_GROUP}_all_cohorts.zip'
shutil.make_archive(str(archive.with_suffix('')),'zip',root_dir=WORK_ROOT,base_dir='results/survey_v3/E4_stability')
print('DONE:',archive)
